# Pipeline completo: Preprocesamiento + Filtrado + Modelo + Optimización + calibrado + store

Este notebook implementa un pipeline end-to-end de Machine Learning:


## PASO 1: Preprocesamiento de datos

In [3]:
from src.preprocessing.base_preprocessing import BasePreprocess

# Instanciamos la clase de preprocesamiento.
# El fichero Excel contiene la lista de variables candidatas a ser predictoras.
base_pre = BasePreprocess("data/variables_withoutExperts.xlsx", "loan_status")

In [4]:
# fit(): aprende los parametros del preprocesamiento SOLO con datos de entrenamiento.
# Esto incluye: categorias del OHE, medianas para imputacion, parametros del QuantileTransformer, etc.
base_pre.fit("data/df_train_small.csv")

/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/src/preprocessing/base_preprocessing.py:62: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  self.train_X_data['earliest_cr_line'] = pd.to_datetime(self.train_X_data['earliest_cr_line'])


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# transform(): aplica las transformaciones aprendidas en fit().
# Devuelve X_train (features) e y_train (target: True=default, False=fully paid).
X_train, y_train = base_pre.transform("data/df_train_small.csv")
print(f"Dimensiones tras preprocesamiento: {X_train.shape[0]} filas x {X_train.shape[1]} columnas")

/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/src/preprocessing/base_preprocessing.py:135: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  X_data['earliest_cr_line'] = pd.to_datetime(X_data['earliest_cr_line'])


Dimensiones tras preprocesamiento: 80000 filas x 2614 columnas


## PASO 2: Filtrado de features

`BaseFiltering` aplica 3 filtros secuenciales:
1. **DropConstantFeatures** (tol=0.9): elimina features donde el 90%+ de valores son iguales
2. **DropCorrelatedFeatures** (threshold=0.8): elimina una de cada par con correlacion > 0.8
3. **ProbeFeatureSelection** (n_probes=10): elimina features menos importantes que ruido aleatorio

In [7]:
from src.filtering.base_filtering import BaseFiltering

# Instanciamos el filtro con los parametros por defecto.
# Todos los parametros son configurables en el constructor.
base_filter = BaseFiltering(
    constant_tol=0.9,
    correlation_threshold=0.8,
    probe_n_probes=10,
    probe_scoring='roc_auc',
    probe_cv=3,
    probe_n_estimators=50,
    probe_max_depth=10
)

In [ ]:
# fit(): aprende que features eliminar usando SOLO datos de train.
# Internamente ejecuta los 3 filtros en secuencia.
base_filter.fit(X_train, y_train)

In [ ]:
# Resumen del filtrado: cuantas features se eliminaron en cada paso.
base_filter.print_summary()

RESUMEN DEL PIPELINE DE FILTRADO
  Features iniciales:              2614
  Eliminadas cuasi-constantes:     -133
  Eliminadas por correlacion:       -1838
  Eliminadas por ProbeFeature:      -480
  Features seleccionadas finales:  163


In [ ]:
# transform(): aplica los filtros aprendidos en fit() a los datos de train.
X_train_filtered = base_filter.transform(X_train)

print(f"Features seleccionadas ({X_train_filtered.shape[1]}):")
print(X_train_filtered.columns.tolist())

Features seleccionadas (163):
['home_ownership_MORTGAGE', 'verification_status_Not Verified', 'term_ 36 months', 'emp_title_00', 'emp_title_01', 'emp_title_02', 'emp_title_04', 'emp_title_06', 'emp_title_10', 'emp_title_13', 'loan_amnt', 'annual_inc', 'dti', 'loan_amnt annual_inc', 'loan_amnt dti', 'loan_amnt inq_last_6mths', 'loan_amnt mths_since_last_delinq', 'loan_amnt open_acc', 'loan_amnt revol_bal', 'loan_amnt revol_util', 'loan_amnt all_util', 'loan_amnt acc_open_past_24mths', 'loan_amnt avg_cur_bal', 'loan_amnt bc_open_to_buy', 'loan_amnt bc_util', 'loan_amnt mo_sin_old_il_acct', 'loan_amnt mo_sin_rcnt_rev_tl_op', 'loan_amnt mo_sin_rcnt_tl', 'loan_amnt mort_acc', 'loan_amnt mths_since_recent_bc', 'loan_amnt mths_since_recent_inq', 'loan_amnt num_actv_rev_tl', 'loan_amnt num_op_rev_tl', 'loan_amnt num_tl_op_past_12m', 'loan_amnt percent_bc_gt_75', 'loan_amnt total_bc_limit', 'loan_amnt earliest_cr_line_year', 'annual_inc dti', 'annual_inc inq_last_6mths', 'annual_inc mths_since_

## PASO 3: Entrenamiento del modelo